---
title: "Report"
format:
  pdf:
    include-in-header:
      text: |
        \usepackage{etoolbox}
        \AtBeginEnvironment{quote}{\footnotesize}
---

# Introduction

Pediatric blunt head trauma is one of the most common reasons children visit the emergency department (ED). Clinicians often need to decide quickly whether a child should receive a head CT to evaluate traumatic brain injury (TBI). CT can be lifesaving, but it also exposes children to ionizing radiation, which may increase long-term risk. This creates a difficult tradeoff: avoid missing clinically important TBIs (ciTBI) while also avoiding unnecessary CT scans for children who are truly low risk.

In this report, I use the PECARN multicenter prospective cohort dataset from 25 EDs in North America to study how bedside clinical features relate to ciTBI. The outcome is ciTBI, defined by serious clinical consequences (death, neurosurgery, prolonged intubation, or TBI-related hospitalization). Because ciTBI is rare, results can be sensitive to preprocessing choices and missing data, so careful cleaning and transparent reporting are important.

This exploratory analysis has two goals. First, I describe what the cleaned dataset looks like and highlight key measurement constraints, especially age-dependent documentation and symptom assessment. Second, I connect these patterns to decision support. I identify feature combinations that stratify risk, and I implement three CT recommendation approaches: the PECARN clinical decision rule (CDR), an L2-regularized logistic regression model, and a calibrated gradient-boosting model. I evaluate them using not only predictive performance, but also calibration, interpretability, and stability under a perturbation that mimics real documentation limitations.

The rest of the report is organized as follows. I first describe the data and key variables, then present the cleaning pipeline and basic exploratory checks. Next, I report three main findings: (1) risk can be concentrated into a high-risk tail using a small set of bedside features, (2) a few “red flag” findings remain the strongest signals after adjustment, and (3) in children under 2 years, “not acting normally” and scalp hematoma together identify a higher-risk subgroup. I then compare the three CT recommendation methods using performance curves, calibration, interpretability, and stability analyses. I end by discussing limitations of the dataset as a representation of clinical reality and the implications for safe CT reduction.

# Data

This report uses the PECARN (Pediatric Emergency Care Applied Research Network) blunt head trauma cohort, collected prospectively across 25 emergency departments in North America. The cohort includes children younger than 18 years who came to the ED within 24 hours after blunt head trauma and had an initial Glasgow Coma Scale (GCS) of 14–15.

The clinical problem is practical: head CT can detect serious brain injury, but CT also exposes children to ionizing radiation. So clinicians want to find children at very low risk for clinically important traumatic brain injury (ciTBI), where CT can be avoided safely.

The dataset records information available during the ED evaluation, including: demographics (especially age), injury mechanism and mechanism severity, symptoms and exam findings (vomiting, loss of consciousness, headache, altered mental status), imaging and disposition variables (CT use, hospitalization).

The outcome in this report is ciTBI, defined as TBI with serious clinical consequences: death, neurosurgery, intubation >24 hours for head trauma, or hospitalization 2 or more nights with TBI findings on CT. In our cleaned analytic dataset, there are 42,412 visits and 376 ciTBI cases (0.9%), which matches the original PECARN analysis cohort.

Because symptom reporting and clinical decisions differ by age, we first summarize the age distribution (Figure 1A). The cohort covers the full pediatric age range, with more patients in younger ages. We also summarize mechanism severity (Figure 1B). This is a key early triage signal in practice, and it helps us understand how many patients fall into low, moderate, and high severity categories before we look at outcomes.

![Figure 1](../figs/fig1.png)

> **Figure 1A. Age distribution of the analytic cohort.** The cohort spans infancy to adolescence. The dashed and solid lines mark the median and mean, showing a younger-skewed distribution.

> **Figure 1B. Mechanism severity (3-level) in the analytic cohort.** Most patients are in the moderate group, with meaningful sample size also in low and high groups; missingness is small.

## Data Collection

The PECARN head trauma data come from a multicenter prospective cohort study in 25 EDs. Patients were enrolled in two phases: a derivation cohort and a validation cohort during 2004–2006. For each patient, clinicians recorded history, injury mechanism, and exam findings using a standardized form before imaging results were known.

Outcome assessment combined multiple sources. When CT was performed, it followed standard clinical workflow and was interpreted by radiologists. Outcomes were also verified using medical record review, and discharged patients had structured follow-up to reduce missed injuries. Overall, this design links bedside predictors to a clinically meaningful outcome without requiring CT for every child.

A key measurement feature is that not all variables are equally available across ages. Some symptoms (headache, amnesia) are difficult or impossible to assess in very young children, so the dataset contains systematic age-related missingness. We show this pattern directly in Figure 2A.

![Figure 2](../figs/fig2.png)

> **Figure 2A. Missingness by age group (selected variables).** Headache and amnesia have very high missingness in ages 0–2, consistent with age-dependent assessment. This missingness should be interpreted as “not measurable / not asked,” not simply random missing data.

> **Figure 2B. Cohort selection and exclusions.** Starting from 43,399 raw records, we removed low-GCS visits and records with missing ciTBI outcome, leaving 42,412 visits for analysis.

## Data Cleaning

Our preprocessing goal was to make the dataset consistent and analysis-ready while keeping as many eligible records as possible. We used a reproducible cleaning script (clean.py) that standardizes variable names, fixes missing-value codes, and resolves internal contradictions in form fields.

1. Eligibility and outcome completeness. We excluded records that did not match the target cohort (low GCS) and records without a ciTBI label (Figure 2B).

2. Standardizing missing values. Some fields used numeric codes for missing or “not applicable”. We converted these to proper missing values (NA) and kept “not applicable” information when it was meaningful.

3. Fixing parent–child inconsistencies. In the raw form, detail fields were sometimes filled even when the parent symptom was “No” (LOC duration filled but LOC history marked “No”). We handled this by keeping the row and setting the contradictory detail fields to NA rather than dropping the visit.

4. Mechanism severity checks. We kept mechanism severity missing when it was missing, and constructed a clean 3-level severity variable only when severity was valid.

5. Outcome consistency check. We re-derived ciTBI from its component outcomes and found no mismatches in the cleaned cohort.

After cleaning, the final dataset has 42,412 observations and 376 ciTBI cases (0.9%), matching the published cohort size.


## Data Exploration

This section gives a basic “first look” at the cleaned cohort before deeper findings. Since ciTBI is rare, it helps to start with (1) cohort structure (age, mechanism severity), and (2) simple outcome rates across major groups. Figure 3 shows ciTBI prevalence by age group and how ciTBI rates change across age × mechanism severity.

![Figure 3](../figs/fig3.png)

> **Figure 3A. ciTBI prevalence by age group (with 95% Wilson intervals).** ciTBI is rare in every age group, and uncertainty is visible because the event count is small.

> **Figure 3B. ciTBI rate by age group × mechanism severity.** Rates are higher in the high-severity mechanism group across ages, which motivates later subgroup comparisons and risk patterns.

# Findings

## Finding 1 — A small set of bedside features concentrates ciTBI risk

ciTBI is rare overall, but risk is not evenly spread across patients. Using a simple logistic model with only bedside variables (age, mechanism severity, vomit, altered mental status, basilar skull fracture signs, scalp hematoma, loss of consciousness history and palpable skull fracture), we can separate children into groups with very different risk levels.

Figure F1A shows a decile-style calibration plot. The observed ciTBI rate increases clearly across higher predicted-risk deciles. Most deciles near the bottom have very low event rates, while the top decile has a much higher rate.

The decile plot shows stratification, but clinicians often ask a more direct question: if we focus on the top x% highest-risk patients, how many ciTBI cases do we capture? Figure F1B answers this using a capture/lift curve. It shows that a relatively small high-risk tail contains a large fraction of ciTBI events, and the lift drops quickly as we include more patients.

![Figure F1](../figs/figf1.png)

> **Figure F1A. Calibration by predicted-risk decile (Wilson 95% CI).** Observed ciTBI rate rises with predicted risk, showing clear stratification using a small set of bedside features.

> **Figure F1B. Risk concentration in the high-risk tail.** “Capture” is the fraction of all ciTBI cases found within the top x% highest predicted risk. “Lift” is enrichment relative to the baseline ciTBI rate.

## Finding 2 — After adjustment, a few “red flags” carry the strongest independent signal

Simple comparisons can be misleading because symptoms often occur together, and some items are documented more consistently than others. To focus on independent associations, we fit a multivariable logistic model and report adjusted odds ratios.

Figure F2A shows that a small set of findings stands out even after adjustment. In particular, fracture-related or neurologic “red flags” (such as palpable skull fracture, altered mental status, basilar skull fracture signs, and high-severity mechanism) have the strongest associations with ciTBI.

Odds ratios describe association, but they do not directly show how much each variable helps prediction when predictors are correlated. So we also compute permutation importance (Figure F2B): we randomly shuffle one feature at a time and measure how much the log loss increases. Features that cause a larger increase add more predictive information.

![Figure F2](../figs/figf2.png)

> **Figure F2A. Adjusted odds ratios (95% CI) from the multivariable logistic model.** A few neurologic/fracture-related “red flags” show the strongest independent associations.

> **Figure F2B. Permutation importance (Δ log loss).** Increase in log loss after permuting each feature; larger values mean the feature contributes more to predictive performance beyond correlated variables.

## Finding 3 — Under age 2, “not acting normally” and scalp hematoma together mark a higher-risk subgroup

Risk factors may not act the same way across ages. For children under 2 years, symptom reporting is limited and documentation has more constraints. In this age group, we see a meaningful interaction between caregiver report “not acting normally” and scalp hematoma/swelling.

Figure F3A shows the under-2 subgroup rates. The group with both factors present has a much higher ciTBI prevalence than groups with only one factor. In contrast, children with neither factor have very low prevalence.

Because clinicians often think in absolute terms, Figure F3B shows absolute risk differences relative to the reference group (“no hematoma × acting normal”), with uncertainty intervals.

![Figure F3](../figs/figf3.png)

> **Figure F3A. Under-2 ciTBI prevalence by hematoma × acting normal.** The combination “hematoma present + not acting normally” has the highest risk, suggesting an interaction in preverbal children.

> **Figure F3B. Absolute risk difference (percentage points) vs the reference subgroup.** Error bars are bootstrap 95% CIs, showing uncertainty in small subgroups.

## Reality Check

As a basic reality check, we compare our cleaned dataset with the published PECARN study. This is mainly to catch major pipeline mistakes.

Our analytic cohort has 42,412 encounters and 376 ciTBI events (~0.9%), matching the headline numbers in the paper. We also recomputed ciTBI from its component outcomes and found no mismatches with the provided ciTBI label. Overall, these checks suggest the cleaning pipeline did not materially change the cohort or outcome.

Most remaining issues are documentation-related rather than structural: we removed low-GCS encounters and records with missing ciTBI label, corrected parent–child inconsistencies by setting incompatible detail fields to missing, and observed a small number of records where mechanism was recorded but severity was missing.

## Stability check

To check whether Finding 1 is stable, we repeat the same modeling and decile calibration after a bootstrap resample of the cohort. This keeps the overall data distribution similar but changes which exact observations appear, so it tests whether the stratification is driven by a small set of influential cases.

Figure S1 compares the decile calibration “before” and “after” bootstrap. Small differences in the top deciles are expected because events are rare, but the main pattern—higher observed ciTBI in higher predicted-risk strata—should remain.

![Figure S1](../figs/figs1.png)

> **Figure S1A. Decile calibration in the original cohort.** 

> **Figure S1B. Decile calibration after one bootstrap resample.** The overall stratification pattern remains similar, with most variation in the high-risk tail.

# Modeling

## Implementation

We compared three ways to recommend head CT, using ciTBI as the target outcome.

1. PECARN clinical decision rule (CDR).
We implemented the published age-stratified rule as a deterministic classifier. If a child did not meet the “very low risk” criteria, we marked the visit as CT recommended.

2. Logistic regression (linear model).
We fit an L2-regularized logistic regression with class balancing. This model outputs a predicted probability of ciTBI.

3. Gradient boosting (non-linear model).
We fit a histogram-based gradient boosting model, and then applied isotonic calibration so the probabilities better match observed event rates.

Preprocessing. Missing values were handled explicitly: numeric variables were imputed with the median; binary/categorical variables were imputed with the most common value; and we added missingness indicators so the models can learn when missingness itself is informative.

Train/test evaluation and threshold choice. Because ciTBI is rare, we report both ROC and precision–recall curves. We chose operating thresholds in a high-sensitivity regime, and then evaluated all models on a held-out test set.

![Figure M1](../figs/figm1.png)

> **Figure M1A. ROC curves on the held-out test set.** Logistic regression and gradient boosting have similar AUROC (≈0.86). The CDR is shown as a single operating point.

> **Figure M1B. Precision–recall curves on the held-out test set.** Because the base rate is very low (~0.9%), precision is low for all models at moderate recall. Average precision is similar for logistic and boosting.

![Figure M2](../figs/figm2.png)

> **Figure M2. Confusion matrices on the test set (CDR vs learned models at chosen operating points).** The learned models were tuned for high sensitivity: logistic misses 1 ciTBI case in this split, while boosting misses 0 but flags more negatives as CT recommended. The CDR is less sensitive here and recommends CT for a smaller fraction of patients.

## Interpretability

These three approaches trade off transparency and flexibility.

1. CDR: fully interpretable. Each recommendation comes from a small set of explicit clinical rules, so a clinician can see exactly which rule triggered the decision.

2. Logistic regression: still fairly interpretable. It uses an additive linear score, so we can summarize the global effects with coefficients (odds ratios). This helps explain which features increase or decrease predicted risk.

3. Boosting: more complex. It can capture non-linearities and interactions, but it is not directly readable. We therefore rely on model-agnostic explanations such as permutation importance, and we also check calibration carefully.

![Figure M3](../figs/figm3.png)

> **Figure M3A. Logistic regression effects (odds ratios).** Variables such as altered mental status and basilar skull fracture signs show strong positive associations, consistent with clinical expectations.

> **Figure M3B. Boosting permutation importance (ΔAP).** Permuting altered mental status causes the largest drop in performance, suggesting it carries the most predictive information for ranking risk.

## Stability

To test robustness, we applied an age-aware perturbation to the test set: we introduced extra missingness in symptom variables to mimic real documentation limits. We then compared: CT recommendation rate, distribution of changes in predicted risk and per-patient risk shifts.

![Figure M4](../figs/figm4.png)

> **Figure M4A. CT recommendation rate before vs after perturbation.** Recommendation rates change only modestly under the perturbation, suggesting the final CT/no-CT decisions are fairly stable.

> **Figure M4B. Distribution of Δ predicted risk (perturbed − original).** Most changes are near zero, but there are occasional larger shifts, especially for the probabilistic models.

![Figure M5](../figs/figm5.png)

> **Figure M5A. Logistic: predicted risk (original vs perturbed).** Many points stay near the diagonal, but some visits show noticeable drops after missingness is added, indicating sensitivity to documentation changes for some profiles.

> **Figure M5B. Boosting: predicted risk (original vs perturbed).** Most visits stay close to the diagonal at low predicted risk, with a few outliers; overall ranking looks more stable for the bulk of cases.

# Discussion

This dataset is large (tens of thousands of visits), but ciTBI is rare. That creates a “rare-event” problem: many subgroup analyses have very small numbers of ciTBI cases, so uncertainty becomes large quickly. It also means performance metrics can look good on ROC while precision remains low, which is why precision–recall plots matter.

Another key issue is that the data are not a perfect mirror of clinical reality. Some variables reflect documentation and patient communication limits, not only physiology. For example, young children cannot reliably report headache or amnesia, so missingness is structured and age-dependent. If we ignore this, we may misread patterns or build models that accidentally learn documentation artifacts.

In this project, the most useful plots are the ones that connect clinical meaning to statistical evidence: (i) risk stratification by simple strata, (ii) calibration and operating points, and (iii) stability under realistic perturbations. These help translate results into a decision setting where the main goal is high sensitivity while avoiding unnecessary CT when possible.

# Conclusion

Using the PECARN cohort, we studied how bedside features relate to ciTBI and how they can support CT decisions.

1. We found that a small set of bedside variables can concentrate risk into a high-risk tail, even though ciTBI is rare overall.
    
2. After adjustment, a few “red flag” findings (neurologic or fracture-related) carry most of the independent signal.
    
3. For children under 2 years, the combination of “not acting normally” and scalp hematoma identifies a higher-risk subgroup, showing that age-specific patterns matter.

In modeling, logistic regression and gradient boosting achieved similar overall discrimination, but all methods face the same practical limit: low prevalence makes precision low at higher recall. Stability checks suggest CT recommendations are not very sensitive to a realistic missingness perturbation, but some predicted probabilities can shift, especially in younger children. For real deployment, external validation across settings and time would still be necessary.

# Academic honesty statement

Professor Bin, I affirm that this report is my own work. Any ideas, code or feedback that came from external sources are clearly acknowledged and properly cited. I did not misrepresent others’ work as mine, and I will include a similar statement with all assignments this semester.

Academic honesty protects trust. It ensures results can be verified and built upon, gives proper credit to others, and keeps evaluation fair for everyone.

# Collaborators

I worked independently.

# Bibliography

Kuppermann, N., et al. (2009). The Lancet, 374(9696), 1160–1170.
